<a href="https://colab.research.google.com/github/ADBarshan/DM_Prediction_in_asymptomatic_people/blob/main/DR_T2DM_scRNAseq_immune_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Single-cell immune-cell analysis of diabetic retinopathy in type 2 diabetes

**Reanalysis of public PBMC scRNA-seq data, with a prediction layer.**

Author: Dr. Anindita Das Barshan
Environment: Google Colab (T4 GPU is optional; this runs on CPU high-RAM)

---

### What this notebook is, and what it is not

This is a **reanalysis of publicly deposited single-cell RNA-seq data**, done to (a) build single-cell competency on immune-cell data and (b) test whether the calibration-and-net-benefit prediction methodology I use on tabular clinical data transfers to single-cell-derived features. It is **not** an original wet-lab study and should not be described as one.

The biological contrast is clinically meaningful: **type 2 diabetes without retinopathy (noDR) versus with diabetic retinopathy (DR)**, in circulating immune cells (PBMCs). The diabetic retinopathy single-cell literature is dominated by immune signals (pro-inflammatory monocyte subsets, T-cell states, microglial and neural-immune interactions), which is why a PBMC immune-cell framing is the right bridge to infection-immunity bioinformatics rather than a retinal-tissue atlas.

### Read this before trusting any result

Two honest limitations govern interpretation throughout and are restated at the end:

1. **Small n.** Public well-phenotyped DR PBMC datasets are tiny (often n=3 per group). Sample-level statistics and any predictive model are **underpowered** and must be read as exploratory, not validated.
2. **Study confound.** If DR and noDR samples come from *different* GEO series, batch effect is confounded with disease status and integration cannot fully separate them. **Prefer a single study that contains both DR and noDR samples.** The two-series example below is a fallback and its differential results are confounded by design.


## 1. Setup

In [ ]:
# Colab install. Pinned-light; let pip resolve compatible versions.
%pip install -q scanpy leidenalg harmonypy scikit-learn scrublet geoparse pydeseq2 2>/dev/null
print("installed")

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, scanpy as sc, anndata as ad
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
sc.settings.verbosity = 1
sc.settings.figdir = "figures"
sc.set_figure_params(dpi=110, frameon=False)
print("scanpy", sc.__version__)

## 2. Sample sheet (edit this)

Everything downstream is driven by this table, so you can use it for a **single mixed study** (preferred) or the **two-series fallback**. One row per sample.

- `sample_id`  unique label
- `condition`  `DR` or `noDR`
- `study`      GEO series (used as an extra batch covariate)
- `path`       folder with 10x output (`matrix.mtx.gz`, `features.tsv.gz`, `barcodes.tsv.gz`) **or** a `.h5` file

Concrete PBMC accessions used together in a recent DR study: **GSE268210** (type 2 diabetes) and **GSE248284** (diabetic retinopathy). Download their supplementary files from GEO, inspect the folder layout, then fill in `path`. The exact filenames on GEO vary, so check before running.


In [ ]:
# --- EDIT: point paths at your downloaded GEO supplementary files ---
sample_sheet = pd.DataFrame([
    # sample_id, condition, study,       path
    ("T2D_1",  "noDR", "GSE268210", "data/GSE268210/T2D_1"),
    ("T2D_2",  "noDR", "GSE268210", "data/GSE268210/T2D_2"),
    ("T2D_3",  "noDR", "GSE268210", "data/GSE268210/T2D_3"),
    ("DR_1",   "DR",   "GSE248284", "data/GSE248284/DR_1"),
    ("DR_2",   "DR",   "GSE248284", "data/GSE248284/DR_2"),
    ("DR_3",   "DR",   "GSE248284", "data/GSE248284/DR_3"),
], columns=["sample_id", "condition", "study", "path"])

same_study = sample_sheet["study"].nunique() == 1
if not same_study:
    print("WARNING: DR and noDR come from different studies. "
          "Batch is confounded with disease status; treat DE and prediction as exploratory only.")
sample_sheet

### 2b. Download helper (Colab has internet)

GEO supplementary files are usually a per-sample tarball of 10x triplets, or per-sample `.h5`. Example pattern (adjust GSM/URLs to what the series actually provides):

```bash
!mkdir -p data/GSE268210 && cd data/GSE268210 && \
 wget -q "ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE268nnn/GSE268210/suppl/GSE268210_RAW.tar" && \
 tar -xf GSE268210_RAW.tar
```

Then arrange each sample's `matrix.mtx.gz` / `features.tsv.gz` / `barcodes.tsv.gz` into its own folder matching `path` above (or keep `.h5` files and point `path` at the file).


## 3. Load and concatenate

In [ ]:
def load_sample(path):
    # Load one sample as AnnData from a 10x mtx folder or a .h5 file.
    if path.endswith(".h5"):
        a = sc.read_10x_h5(path)
    else:
        a = sc.read_10x_mtx(path, var_names="gene_symbols", cache=True)
    a.var_names_make_unique()
    return a

adatas = {}
for _, r in sample_sheet.iterrows():
    a = load_sample(r["path"])
    a.obs["sample"]    = r["sample_id"]
    a.obs["condition"] = r["condition"]
    a.obs["study"]     = r["study"]
    adatas[r["sample_id"]] = a

adata = ad.concat(adatas, join="inner", label="sample_key", index_unique="-")
adata.obs_names_make_unique()
adata.layers["counts"] = adata.X.copy()
print(adata)

## 4. Quality control

Per-cell filtering on detected genes, total counts, and mitochondrial fraction. Thresholds are starting points; inspect the distributions and adjust per dataset rather than copying blindly.


In [ ]:
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, inplace=True)

sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
             jitter=0.4, groupby="sample", rotation=90, save="_qc.png")

MIN_GENES, MAX_GENES, MAX_MT = 200, 6000, 15   # <-- tune to your data
before = adata.n_obs
adata = adata[(adata.obs.n_genes_by_counts > MIN_GENES) &
              (adata.obs.n_genes_by_counts < MAX_GENES) &
              (adata.obs.pct_counts_mt < MAX_MT)].copy()
print(f"cells: {before} -> {adata.n_obs}")

## 5. Doublet removal (per sample)

Scrublet is run within each sample, because doublet rates are sample-specific.


In [ ]:
sc.pp.scrublet(adata, batch_key="sample")
adata = adata[~adata.obs["predicted_doublet"]].copy()
print("after doublet removal:", adata.n_obs)

## 6. Normalisation, HVGs, PCA

Counts are preserved in `.layers['counts']` and `.raw` for later pseudobulk and gene scoring. HVGs are selected with `batch_key='sample'` so no single sample dominates.


In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata

sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key="sample")
adata_hvg = adata[:, adata.var.highly_variable].copy()
sc.pp.scale(adata_hvg, max_value=10)
sc.tl.pca(adata_hvg, n_comps=30, random_state=SEED)
sc.pl.pca_variance_ratio(adata_hvg, n_pcs=30, save="_pca_var.png")

## 7. Batch integration (Harmony)

Called directly (orientation-safe across harmonypy versions). Integration corrects technical variation between samples; **it does not fix the study confound** if cases and controls are from different series.


In [ ]:
import harmonypy
def harmony_integrate(a, batch_keys, basis="X_pca", out="X_pca_harmony"):
    ho = harmonypy.run_harmony(a.obsm[basis], a.obs, batch_keys)
    Z = np.asarray(ho.Z_corr)
    if Z.shape[0] != a.n_obs:        # ensure cells on axis 0
        Z = Z.T
    a.obsm[out] = Z
    return a

batch_keys = ["sample"] if same_study else ["study", "sample"]
harmony_integrate(adata_hvg, batch_keys)
print("integrated rep:", adata_hvg.obsm["X_pca_harmony"].shape)

In [ ]:
sc.pp.neighbors(adata_hvg, use_rep="X_pca_harmony", n_neighbors=15, random_state=SEED)
sc.tl.umap(adata_hvg, random_state=SEED)
sc.tl.leiden(adata_hvg, resolution=0.5, flavor="igraph", n_iterations=2,
             directed=False, random_state=SEED)
sc.pl.umap(adata_hvg, color=["leiden", "sample", "condition"], wspace=0.4, save="_clusters.png")
print("clusters:", adata_hvg.obs["leiden"].nunique())

## 8. Cell-type annotation

Score canonical PBMC marker sets per cluster and label each cluster by its top-scoring set. Inspect the dotplot and rename clusters manually where biology demands; automated labelling is a first pass, not the final word.


In [ ]:
marker_sets = {
    "CD4 T":      ["CD3D", "CD3E", "IL7R", "CCR7"],
    "CD8 T":      ["CD3D", "CD8A", "GZMK"],
    "NK":         ["NKG7", "GNLY", "KLRD1"],
    "B":          ["MS4A1", "CD79A", "CD79B"],
    "CD14 Mono":  ["CD14", "LYZ", "S100A8", "S100A9"],
    "CD16 Mono":  ["FCGR3A", "MS4A7"],
    "DC":         ["FCER1A", "CST3", "CLEC9A"],
    "Platelet":   ["PPBP", "PF4"],
}
score_cols = []
for name, genes in marker_sets.items():
    genes = [g for g in genes if g in adata_hvg.raw.var_names]
    col = f"score_{name}"
    sc.tl.score_genes(adata_hvg, genes, score_name=col, use_raw=True)
    score_cols.append(col)

cluster_scores = adata_hvg.obs.groupby("leiden")[score_cols].mean()
label_map = cluster_scores.idxmax(axis=1).str.replace("score_", "")
adata_hvg.obs["cell_type"] = adata_hvg.obs["leiden"].map(label_map).astype("category")

flat = [g for gs in marker_sets.values() for g in gs if g in adata_hvg.raw.var_names]
sc.pl.dotplot(adata_hvg, flat, groupby="cell_type", use_raw=True, save="_markers.png")
adata_hvg.obs["cell_type"].value_counts()

## 9. Cell-type composition by condition

A shift in immune-cell proportions (for example a higher CD14 monocyte fraction in DR) is one of the clearest single-cell readouts. With few samples, treat this descriptively. For a proper compositional test with uncertainty, scCODA is the right tool; noted here rather than over-claimed on n=6.


In [ ]:
comp = pd.crosstab(adata_hvg.obs["sample"], adata_hvg.obs["cell_type"], normalize="index")
comp["condition"] = sample_sheet.set_index("sample_id").loc[comp.index, "condition"].values
print(comp.round(3))

prop = comp.drop(columns="condition")
prop.plot(kind="bar", stacked=True, figsize=(8, 4))
plt.ylabel("fraction of cells"); plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.savefig("figures/composition.png", dpi=110); plt.show()

## 10. Differential expression (pseudobulk, per cell type)

Per-cell DE across conditions treats cells as independent and **inflates significance**; the correct unit is the sample. We aggregate counts to pseudobulk per (sample, cell type) and run DESeq2 (pydeseq2) within each cell type. With n=3 per group, power is low, so read effect sizes and direction rather than chasing p-values.

If DR and noDR are from different studies, these results are confounded with study and should not be reported as disease effects.


In [ ]:
# pseudobulk: sum raw counts per sample x cell_type
adata_counts = adata_hvg.copy()
adata_counts.X = adata_counts.raw[:, adata_counts.var_names].X
pb = sc.get.aggregate(adata_counts, by=["sample", "cell_type"], func="sum")
pb_df = pd.DataFrame(pb.layers["sum"], index=pb.obs_names, columns=pb.var_names)
pb_meta = pb.obs.copy()
pb_meta["condition"] = sample_sheet.set_index("sample_id").loc[pb_meta["sample"], "condition"].values
print("pseudobulk table:", pb_df.shape)

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

def de_for_celltype(ct, min_samples_per_group=2):
    m = pb_meta[pb_meta["cell_type"] == ct]
    counts = pb_df.loc[m.index].round().astype(int)
    counts = counts.loc[:, counts.sum(0) > 10]           # drop near-zero genes
    if m["condition"].value_counts().min() < min_samples_per_group:
        return None
    dds = DeseqDataSet(counts=counts, metadata=m[["condition"]],
                       design="~condition", quiet=True)
    dds.deseq2()
    st = DeseqStats(dds, contrast=["condition", "DR", "noDR"], quiet=True)
    st.summary()
    return st.results_df.sort_values("padj")

target_ct = "CD14 Mono"    # inflammatory monocytes are the DR-relevant population
res = de_for_celltype(target_ct) if target_ct in pb_meta["cell_type"].values else None
if res is not None:
    print(f"Top DR-vs-noDR genes in {target_ct}:")
    print(res.head(15)[["log2FoldChange", "pvalue", "padj"]])
else:
    print("Not enough samples per group for DESeq2 in this cell type.")

## 11. Immune signature + prediction (honest, small-n)

This is the layer that connects to my clinical-prediction work: derive an immune-cell signature, summarise it per sample, and test whether it separates DR from noDR under **leave-one-sample-out** cross-validation. With ~6 samples this **cannot** be a validated model. I run it, report the leave-one-out AUC, and state plainly that it is a proof-of-transfer of methodology, not evidence of a working classifier.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import roc_auc_score

# signature: inflammatory monocyte programme (edit from your DE results)
infl_sig = [g for g in ["CD14", "LYZ", "S100A8", "S100A9", "FCN1"]
            if g in adata_hvg.raw.var_names]
sc.tl.score_genes(adata_hvg, infl_sig, score_name="infl_mono_sig", use_raw=True)

# per-sample features: mean signature + monocyte fraction
feat = adata_hvg.obs.groupby("sample").agg(
    infl=("infl_mono_sig", "mean"),
    mono_frac=("cell_type", lambda s: s.astype(str).str.contains("Mono").mean()),
)
feat["y"] = (sample_sheet.set_index("sample_id").loc[feat.index, "condition"] == "DR").astype(int).values

X = feat[["infl", "mono_frac"]].values
y = feat["y"].values
groups = feat.index.values

logo, preds = LeaveOneGroupOut(), np.zeros(len(y))
for tr, te in logo.split(X, y, groups):
    preds[te] = LogisticRegression().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]

try:
    auc = roc_auc_score(y, preds)
    msg = f"leave-one-sample-out AUC = {auc:.2f}"
except ValueError:
    msg = "AUC undefined (a fold had one class only)"

print(feat.round(3))
print(msg)
print(f"n_samples = {len(y)}  -> UNDERPOWERED. Exploratory only; do not report as a validated model.")

## 12. Limitations (state these in any write-up)

1. **Sample size.** ~3 per group. All sample-level statistics and the prediction layer are exploratory. External validation on an independent cohort is required before any clinical claim.
2. **Study confound.** Where DR and noDR come from separate GEO series, disease status is confounded with study/batch; Harmony reduces but cannot remove this. A single study containing both groups is strongly preferred.
3. **Compartment.** PBMCs are a peripheral proxy; they do not capture retinal-resident microglia or the local neural-immune interactions central to DR pathology.
4. **Annotation.** Marker-based labels are a first pass and were reviewed manually; rare or transitional states may be merged.
5. **Reanalysis, not new data.** This reuses deposited data to build single-cell competence and test methodology transfer.


## 13. Reproducibility

In [ ]:
import session_info
try:
    session_info.show()
except Exception:
    import sys; print("python", sys.version); print("scanpy", sc.__version__)
# adata_hvg.write("dr_pbmc_processed.h5ad")   # uncomment to save the processed object

## 14. How to describe this honestly in the PhD email

> As a first step into omics, I reanalysed public PBMC single-cell data from type 2 diabetes with and without retinopathy (e.g. GSE268210, GSE248284): QC and doublet removal, Harmony integration, Leiden clustering and marker-based annotation, pseudobulk differential expression per cell type, and an inflammatory-monocyte signature. I then tested whether my calibration-and-net-benefit prediction approach transfers to single-cell-derived features under leave-one-sample-out validation, and I am explicit in the notebook that the public sample size makes this exploratory. The notebook is here: [link].

This claims exactly what was done, flags the small-n honestly, and shows the reviewer the methodology transfer without overselling. Adjust the signature genes and the one-line finding to match your actual DE output before sending.
